# Traffic-volume anomaly detection

Completed classical experiment aligned to the final IFN737 2C report and aligned 2B presentation. This detects unusual observed volume patterns; it does not forecast future counts or establish crash causes.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
from detect import detect
assert Path("protocol.json").exists(), "Open this notebook from traffic_anomaly_detection"
protocol = json.loads(Path("protocol.json").read_text())
selected = json.loads(Path("selected_parameters.json").read_text())
print(selected)

## Frozen allocation and input data
The procedure is selected using development validation data; final sections still receive their own earlier local fitting and validation calibration.

In [ ]:
sites = pd.DataFrame(protocol["sites"])
display(sites.groupby("role").size())
display(pd.read_csv("data/0185-E.csv").head())
print("Folds:", protocol["folds"])

## Final-site detection metrics
Average seeds and folds within a section, then average the 117 final sections equally. Inspector set 2 reuses thresholds calibrated on inspector set 1.

In [ ]:
summary = pd.read_csv("results/summary.csv")
display(summary.query("role == 'final' and label_set == 'label_set_1'")[["model", "f1", "precision", "recall", "false_alerts_per_observed_day", "average_precision"]])
display(Image(filename="results/final_f1.png"))

## Matched effects and sensitivity
PCA reconstruction versus MAD does not isolate compression. These paired IF comparisons do. Intervals resample sites after within-site averaging; spatial dependence remains a limitation.

In [ ]:
display(pd.read_csv("results/paired_effects.csv"))
display(pd.read_csv("results/normal_only_sensitivity.csv"))
display(Image(filename="results/paired_effects.png"))

## Cost, precision–recall and errors
The serial isolated benchmark uses the same fixed section and fold for each representation. Error groups overlap.

In [ ]:
display(pd.read_csv("results/isolated_benchmark.csv").groupby("model").mean(numeric_only=True))
display(pd.read_csv("results/error_groups.csv").head(12))
display(Image(filename="results/precision_recall.png"))
display(Image(filename="results/example_detection.png"))

## Score the supplied historical example
This checkpoint was chosen by development validation results and locally fitted/calibrated on section 0185-E. The partition column distinguishes in-sample, validation and later observations. No live feed is connected.

In [ ]:
result = detect("models/0185-E/detector.joblib", "data/0185-E.csv")
display(result.tail())

## Train another section or reproduce
Commands are shown without executing automatically. Run them intentionally in a separate results copy if you want to preserve the delivered experiment.

In [ ]:
# import subprocess, sys
# subprocess.run([sys.executable, "train_site.py", "--data", "data/2414-E.csv", "--out", "models/2414-E"], check=True)
# subprocess.run([sys.executable, "run_all.py"], check=True)
print(json.loads(Path("results/verification.json").read_text()))